# Multinomial Logistic Regression Error Analysis

## Brain Connectivity Classification: Establishing Baseline Performance and Motivating Alternative Approaches

---

### Research Context

This notebook presents a systematic error analysis of multinomial logistic regression applied to brain region classification using functional connectivity fingerprints. The analysis forms a critical component of the "error-as-signal" framework: by training classifiers on resting-state fMRI data and testing on task-based data, classification errors serve as indicators of functional brain reorganization during cognitive engagement.

**The central question**: Do the error patterns of multinomial classification reveal systematic limitations that would be addressed by One-vs-Rest (OvR) or One-vs-One (OvO) strategies?

### Hypotheses

If multinomial logistic regression has fundamental limitations for this 232-class problem, we would expect to observe:

1. **Diluted discriminative power**: Errors distributed broadly across many classes rather than concentrated in interpretable patterns
2. **Cross-boundary confusion**: Majority of errors crossing both network AND hemisphere boundaries (suggesting the joint classification problem is too complex)
3. **Uniform degradation**: Similar error rate increases across all networks during task transfer (rather than task-specific patterns)
4. **Probability diffusion**: Low confidence predictions spread across many candidate classes

### Notebook Structure

1. **Setup & Data Loading** — Environment configuration and data import
2. **Methods Summary** — Brief overview of data, parcellation, and protocol
3. **Model Performance Overview** — Cross-validation and task transfer accuracy
4. **Error Taxonomy Analysis** — Categorizing errors by hemisphere and network boundaries
5. **Network-Level Error Patterns** — Which networks confuse with which?
6. **Hemispheric Flow Analysis** — Sankey diagrams of error pathways
7. **The Case Against Multinomial** — Synthesis of limitations and motivation for OvR/OvO
8. **Conclusions** — Summary and implications for subsequent analyses

---
## 1. Setup & Data Loading

In [3]:
# Core Libraries
import numpy as np
import pandas as pd
import json
from pathlib import Path
import warnings

# Statistical Analysis
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import confusion_matrix

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Configuration
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [4]:
# Path Configuration
PROJECT_ROOT = Path('/home/sjoon/projects/brain_connectivity_classifier')
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'

# Directory Setup for Each Model
full_multi_dir = RESULTS_DIR / 'full_connectivity_analysis' / 'multinomial'
LH_multi_dir = RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'multinomial'
RH_multi_dir = RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'multinomial'

full_task_dir = RESULTS_DIR / 'full_connectivity_analysis' / 'task_testing'
LH_task_dir = RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'task_testing_multinomial'
RH_task_dir = RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'task_testing_multinomial'

print(f"Results Directory: {RESULTS_DIR}")

Results Directory: /home/sjoon/projects/brain_connectivity_classifier/data/results


In [5]:
# Helper Functions
def load_json(filepath):
    """Load JSON file safely"""
    with open(filepath, 'r') as f:
        return json.load(f)

def load_npy(filepath):
    """Load numpy file safely"""
    return np.load(filepath, allow_pickle=True)

def load_csv(filepath):
    """Load CSV file safely"""
    return pd.read_csv(filepath)

def load_bundle(m_dir, t_dir):
    """Load standard file patterns from directories."""
    return [
        load_json(m_dir / 'overall_metrics.json'), 
        load_json(m_dir / 'fold_metrics.json'),
        load_csv(m_dir / 'network_metrics.csv'), 
        load_csv(m_dir / 'per_region_metrics.csv'),
        load_npy(m_dir / 'confusion_matrix.npy'), 
        load_npy(m_dir / 'cv_predictions.npy'),
        load_npy(m_dir / 'cv_probabilities.npy'), 
        load_npy(m_dir / 'cv_true_labels.npy'),
        load_npy(m_dir / 'cv_fold_indices.npy'), 
        load_json(t_dir / 'task_testing_summary.json'),
        load_csv(t_dir / 'task_network_metrics.csv'), 
        load_csv(t_dir / 'task_per_region_metrics.csv'),
        load_npy(t_dir / 'task_confusion_matrix.npy'), 
        load_npy(t_dir / 'task_predictions.npy'),
        load_npy(t_dir / 'task_probabilities.npy'), 
        load_npy(t_dir / 'task_true_labels.npy')
    ]

print("✓ Helper functions loaded")

✓ Helper functions loaded


In [6]:
# Load All Data

# Full Connectivity Model (232 regions)
overall_metrics_full, fold_metrics_full, network_metrics_cv_full, per_region_metrics_cv_full, \
confusion_matrix_cv_full, cv_predictions_full, cv_probabilities_full, cv_true_labels_full, \
cv_fold_indices_full, task_summary_full, network_metrics_task_full, per_region_metrics_task_full, \
confusion_matrix_task_full, task_predictions_full, task_probabilities_full, task_true_labels_full = load_bundle(full_multi_dir, full_task_dir)

region_info_full = load_csv(full_multi_dir / 'region_info.csv')

# Left Hemisphere Model (116 regions)
overall_metrics_lh, fold_metrics_lh, network_metrics_cv_lh, per_region_metrics_cv_lh, \
confusion_matrix_cv_lh, cv_predictions_lh, cv_probabilities_lh, cv_true_labels_lh, \
cv_fold_indices_lh, task_summary_lh, network_metrics_task_lh, per_region_metrics_task_lh, \
confusion_matrix_task_lh, task_predictions_lh, task_probabilities_lh, task_true_labels_lh = load_bundle(LH_multi_dir, LH_task_dir)

region_info_lh = region_info_full[region_info_full['hemisphere'] == 'left'].copy().assign(region_idx=lambda x: range(len(x)))

# Right Hemisphere Model (116 regions)
overall_metrics_rh, fold_metrics_rh, network_metrics_cv_rh, per_region_metrics_cv_rh, \
confusion_matrix_cv_rh, cv_predictions_rh, cv_probabilities_rh, cv_true_labels_rh, \
cv_fold_indices_rh, task_summary_rh, network_metrics_task_rh, per_region_metrics_task_rh, \
confusion_matrix_task_rh, task_predictions_rh, task_probabilities_rh, task_true_labels_rh = load_bundle(RH_multi_dir, RH_task_dir)

region_info_rh = region_info_full[region_info_full['hemisphere'] == 'right'].copy().assign(region_idx=lambda x: range(len(x)))

print("✓ All data loaded successfully")

✓ All data loaded successfully


In [7]:
# 8-Network Grouping Mapping (Yeo-7 + Subcortical)
# This mapping aggregates 33 fine-grained networks into 8 major functional systems
# for interpretable analysis while preserving neurobiological meaning.

network_to_major = {
    'VisCent': 'Visual', 'VisPeri': 'Visual',
    'SomMotA': 'Somatomotor', 'SomMotB': 'Somatomotor',
    'DorsAttnA': 'Dorsal Attention', 'DorsAttnB': 'Dorsal Attention',
    'SalVentAttnA': 'Salience/Ventral Attention', 'SalVentAttnB': 'Salience/Ventral Attention',
    'LimbicA': 'Limbic', 'LimbicB': 'Limbic',
    'ContA': 'Control', 'ContB': 'Control', 'ContC': 'Control',
    'DefaultA': 'Default', 'DefaultB': 'Default', 'DefaultC': 'Default',
    'TempPar': 'Default',
    'Hippocampus_ant': 'Subcortical', 'Hippocampus_post': 'Subcortical',
    'Amygdala_lat': 'Subcortical', 'Amygdala_med': 'Subcortical',
    'Thalamus_DA': 'Subcortical', 'Thalamus_DP': 'Subcortical',
    'Thalamus_VA': 'Subcortical', 'Thalamus_VP': 'Subcortical',
    'Caudate_ant': 'Subcortical', 'Caudate_post': 'Subcortical',
    'Putamen_ant': 'Subcortical', 'Putamen_post': 'Subcortical',
    'Pallidum_ant': 'Subcortical', 'Pallidum_post': 'Subcortical',
    'Accumbens_core': 'Subcortical', 'Accumbens_shell': 'Subcortical'
}

# Apply mapping to all region info DataFrames
for df in [region_info_full, region_info_lh, region_info_rh]:
    df['major_network'] = df['network'].map(network_to_major)

# Define canonical network order for consistent visualization
NETWORKS = ['Visual', 'Somatomotor', 'Dorsal Attention', 'Salience/Ventral Attention',
            'Limbic', 'Control', 'Default', 'Subcortical']

# Define sample sizes for rate calculations
N_SUBJECTS = 224
N_REGIONS_FULL = 232
N_REGIONS_HEMI = 116
TOTAL_SAMPLES_FULL = N_SUBJECTS * N_REGIONS_FULL  # 51,968
TOTAL_SAMPLES_HEMI = N_SUBJECTS * N_REGIONS_HEMI  # 25,984

print("✓ Network mappings and constants defined")

✓ Network mappings and constants defined


---
## 2. Methods Summary

### Data
- **Training**: AOMIC PIOP-2 resting-state fMRI (N=224 subjects)
- **Testing**: AOMIC PIOP-1 Gender Stroop task fMRI (N=200 subjects)

### Parcellation
- **Cortical**: Schaefer 200-region atlas (7-network version)
- **Subcortical**: Tian Scale I atlas (32 regions)
- **Total**: 232 brain regions (116 per hemisphere)

### Classification Protocol
1. Extract functional connectivity fingerprints (231 features per region)
2. Train multinomial logistic regression with 5-fold cross-validation on resting-state data
3. Test trained model on task-based data without retraining
4. Analyze error patterns as indicators of functional reorganization

### Model Configurations
- **Full (232)**: All regions, including cross-hemispheric connections
- **Left Hemisphere (116)**: Intra-hemispheric connections only
- **Right Hemisphere (116)**: Intra-hemispheric connections only

---
## 3. Model Performance Overview

We first establish baseline performance metrics before examining error patterns. The key comparison is between cross-validation accuracy (resting-state) and task transfer accuracy.

**Expected finding**: If the multinomial model captures stable fingerprints, task accuracy should be close to CV accuracy. Large drops suggest the model is sensitive to state-dependent changes in connectivity.

In [8]:
# Build comprehensive model comparison table

models = [
    ('Full (232)', overall_metrics_full, task_summary_full, region_info_full, 
     cv_predictions_full, task_predictions_full, confusion_matrix_cv_full),
    ('Left (116)', overall_metrics_lh, task_summary_lh, region_info_lh, 
     cv_predictions_lh, task_predictions_lh, confusion_matrix_cv_lh),
    ('Right (116)', overall_metrics_rh, task_summary_rh, region_info_rh, 
     cv_predictions_rh, task_predictions_rh, confusion_matrix_cv_rh)
]

# Calculate pooled metrics for Combined (L+R)
combined_cv_total = len(cv_predictions_lh) + len(cv_predictions_rh)
combined_cv_correct = (int(overall_metrics_lh['accuracy'] * len(cv_predictions_lh)) + 
                       int(overall_metrics_rh['accuracy'] * len(cv_predictions_rh)))
combined_cv_acc = combined_cv_correct / combined_cv_total

combined_task_total = len(task_predictions_lh) + len(task_predictions_rh)
combined_task_correct = (int(task_summary_lh['task_test_accuracy'] * len(task_predictions_lh)) + 
                         int(task_summary_rh['task_test_accuracy'] * len(task_predictions_rh)))
combined_task_acc = combined_task_correct / combined_task_total
combined_rest_train = (task_summary_lh['rest_train_accuracy'] + task_summary_rh['rest_train_accuracy']) / 2

# Build comparison DataFrame
rows = []
for m_name, m_metrics, m_task, m_regions, m_cv_p, m_task_p, m_cm in models:
    train_acc = m_task['rest_train_accuracy']
    task_acc = m_task['task_test_accuracy']
    cv_acc = m_metrics['accuracy']
    
    rows.append({
        'Model': m_name,
        'CV_Accuracy': cv_acc,
        'Rest_Train_Acc': train_acc,
        'Task_Accuracy': task_acc,
        'Accuracy_Drop': train_acc - task_acc,
        'Generalization': task_acc / cv_acc,
        'N_Regions': len(m_regions)
    })

rows.append({
    'Model': 'Combined (L+R)',
    'CV_Accuracy': combined_cv_acc,
    'Rest_Train_Acc': combined_rest_train,
    'Task_Accuracy': combined_task_acc,
    'Accuracy_Drop': combined_rest_train - combined_task_acc,
    'Generalization': combined_task_acc / combined_cv_acc,
    'N_Regions': 232
})

comparison_df = pd.DataFrame(rows)

# Display formatted results
print("="*100)
print("MODEL PERFORMANCE COMPARISON: RESTING-STATE TRAINING vs. TASK TESTING")
print("="*100)
print(f"\n{'Model':<15} {'CV Acc':>10} {'Train Acc':>12} {'Task Acc':>10} {'Drop':>10} {'Gen. Ratio':>12}")
print("-"*70)
for _, row in comparison_df.iterrows():
    print(f"{row['Model']:<15} {row['CV_Accuracy']:>10.2%} {row['Rest_Train_Acc']:>12.2%} "
          f"{row['Task_Accuracy']:>10.2%} {row['Accuracy_Drop']:>10.2%} {row['Generalization']:>12.2%}")

print("\n" + "="*100)
print("KEY OBSERVATIONS:")
print("="*100)
print(f"• Full model achieves {comparison_df.iloc[0]['CV_Accuracy']:.1%} CV accuracy across 232 classes")
print(f"• Task transfer drops accuracy by {comparison_df.iloc[0]['Accuracy_Drop']:.1%} (Full) vs {comparison_df.iloc[3]['Accuracy_Drop']:.1%} (Combined)")
print(f"• The Full model shows {(comparison_df.iloc[3]['Accuracy_Drop'] - comparison_df.iloc[0]['Accuracy_Drop'])*100:.2f}% greater stability")
print(f"• This suggests cross-hemispheric features provide meaningful classification signal")
print("="*100)

MODEL PERFORMANCE COMPARISON: RESTING-STATE TRAINING vs. TASK TESTING

Model               CV Acc    Train Acc   Task Acc       Drop   Gen. Ratio
----------------------------------------------------------------------
Full (232)          92.41%       99.02%     89.24%      9.78%       96.57%
Left (116)          91.82%       97.73%     86.93%     10.80%       94.68%
Right (116)         91.36%       97.74%     86.31%     11.43%       94.47%
Combined (L+R)      91.59%       97.74%     86.62%     11.12%       94.58%

KEY OBSERVATIONS:
• Full model achieves 92.4% CV accuracy across 232 classes
• Task transfer drops accuracy by 9.8% (Full) vs 11.1% (Combined)
• The Full model shows 1.34% greater stability
• This suggests cross-hemispheric features provide meaningful classification signal


In [74]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Prepare Data
df = comparison_df.copy()

# 2. Create Figure with Dual Y-Axes
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add CV Accuracy (Baseline performance)
fig.add_trace(
    go.Bar(
        x=df['Model'], y=df['CV_Accuracy'],
        name='CV Accuracy (Rest)', marker_color='#2b8cbe', opacity=0.7,
        text=[f"{val:.1%}" for val in df['CV_Accuracy']], textposition='auto',
        hovertemplate="<b>%{x}</b><br>CV Accuracy: %{y:.1%}<extra></extra>"
    ), secondary_y=False
)

# Add Task Accuracy (Transfer performance)
fig.add_trace(
    go.Bar(
        x=df['Model'], y=df['Task_Accuracy'],
        name='Task Accuracy', marker_color='#e6550d', opacity=0.8,
        text=[f"{val:.1%}" for val in df['Task_Accuracy']], textposition='auto',
        hovertemplate="<b>%{x}</b><br>Task Accuracy: %{y:.1%}<extra></extra>"
    ), secondary_y=False
)

# Add Generalization Ratio (Line plot on secondary axis)
fig.add_trace(
    go.Scatter(
        x=df['Model'], y=df['Generalization'],
        name='Generalization Ratio', mode='lines+markers+text',
        line=dict(color='#333', width=3, dash='dot'),
        marker=dict(size=12, symbol='diamond', line=dict(width=2, color='white')),
        text=[f"{val:.1%}" for val in df['Generalization']], textposition='top center',
        hovertemplate="<b>%{x}</b><br>Gen. Ratio: %{y:.1%}<extra></extra>"
    ), secondary_y=True
)

# 3. Layout Customization
fig.update_layout(
    title=dict(
        text="<b>Multinomial Model Performance and Cross-State Transferability</b><br>" +
             "<sup>Comparison of model performance at Rest vs. transfer to Task conditions</sup>",
        x=0.5, font=dict(size=20)
    ),
    template='plotly_white',
    height=600, width=1100,
    barmode='group',
    legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="left", x=0.5),
    margin=dict(t=150, b=80, l=80, r=80)
)

# Axis Formatting
fig.update_yaxes(title_text="<b>Classification Accuracy</b>", tickformat=".0%", secondary_y=False, range=[0, 1.1])
fig.update_yaxes(title_text="<b>Generalization Ratio (Task / CV)</b>", tickformat=".0%", secondary_y=True, range=[0, 1.1])
fig.update_xaxes(title_text="<b>Model Strategy</b>")

# Add Highlight Annotation for the Full Model Stability
fig.add_annotation(
    x='Full (232)', y=df.loc[df['Model'] == 'Full (232)', 'Task_Accuracy'].values[0],
    text="Highest Stability<br>(Bilateral Features)",
    showarrow=True, arrowhead=2, ax=0, ay=-100,
    bgcolor="rgba(255, 255, 255, 0.9)", bordercolor="#2b8cbe"
)

fig.show()

---
## 4. Error Taxonomy Analysis

To understand *where* the multinomial model struggles, we categorize errors along two dimensions:

1. **Hemisphere**: Did the error stay within the same hemisphere or cross to the opposite?
2. **Network**: Did the error stay within the same functional network or cross to a different one?

This creates four error categories:
- **Within-Network**: Same hemisphere AND same network (fine-grained confusion between nearby regions)
- **Cross-Network Only**: Same hemisphere, different network (network boundary crossing)
- **Cross-Hemisphere Only**: Different hemisphere, same network (hemispheric homolog confusion)
- **Both Crossed**: Different hemisphere AND different network (most severe confusion)

**Hypothesis test**: If errors are primarily within-network, this suggests the model captures network-level structure well. If errors are predominantly cross-boundary, the multinomial approach may be overwhelmed by the 232-class complexity.

In [9]:
def analyze_errors_comprehensive(true_labels, pred_labels, region_info_df, strategy_name, data_type):
    """Analyze classification errors by hemisphere and network confusion types."""
    err_mask = true_labels != pred_labels
    total_samples, total_err = len(true_labels), err_mask.sum()
    
    if total_err == 0:
        return {'Model': strategy_name, 'Data': data_type, 'Total_Errors': 0, 'Error_Rate': 0,
                'Within_Network': 0, 'Cross_Network': 0, 'Cross_Hemi': 0, 'Both_Crossed': 0}

    lookup = region_info_df.set_index('region_idx')
    df = pd.DataFrame({'true_idx': true_labels[err_mask], 'pred_idx': pred_labels[err_mask]})
    
    for col, attr in [('true_net', 'major_network'), ('pred_net', 'major_network'), 
                      ('true_hemi', 'hemisphere'), ('pred_hemi', 'hemisphere')]:
        df[col] = df['true_idx' if 'true' in col else 'pred_idx'].map(lookup[attr])

    same_hemi = df['true_hemi'] == df['pred_hemi']
    same_net = df['true_net'] == df['pred_net']

    return {
        'Model': strategy_name, 
        'Data': data_type, 
        'Total_Errors': total_err,
        'Error_Rate': total_err / total_samples,
        'Within_Network': (same_hemi & same_net).sum(),
        'Cross_Network': (same_hemi & ~same_net).sum(),
        'Cross_Hemi': (~same_hemi & same_net).sum(),
        'Both_Crossed': (~same_hemi & ~same_net).sum()
    }

# Configure analyses
configs = [
    (cv_true_labels_full, cv_predictions_full, region_info_full, "Full", "Rest"),
    (task_true_labels_full, task_predictions_full, region_info_full, "Full", "Task"),
    (cv_true_labels_lh, cv_predictions_lh, region_info_lh, "LH", "Rest"),
    (task_true_labels_lh, task_predictions_lh, region_info_lh, "LH", "Task"),
    (cv_true_labels_rh, cv_predictions_rh, region_info_rh, "RH", "Rest"),
    (task_true_labels_rh, task_predictions_rh, region_info_rh, "RH", "Task")
]

# Process all configurations
error_results = [analyze_errors_comprehensive(*c) for c in configs]
error_df = pd.DataFrame(error_results)

# Calculate percentages
for col in ['Within_Network', 'Cross_Network', 'Cross_Hemi', 'Both_Crossed']:
    error_df[f'{col}_Pct'] = (error_df[col] / error_df['Total_Errors'] * 100).round(1)

# Display results
print("="*110)
print("ERROR TAXONOMY: Where Do Classification Errors Occur?")
print("="*110)
print(f"\n{'Model':<8} {'Data':<6} {'Errors':>8} {'Rate':>8} {'Within-Net':>14} {'Cross-Net':>14} {'Cross-Hemi':>14} {'Both':>14}")
print("-"*100)

for _, row in error_df.iterrows():
    print(f"{row['Model']:<8} {row['Data']:<6} {row['Total_Errors']:>8} {row['Error_Rate']:>7.1%} "
          f"{row['Within_Network']:>6} ({row['Within_Network_Pct']:>5.1f}%) "
          f"{row['Cross_Network']:>6} ({row['Cross_Network_Pct']:>5.1f}%) "
          f"{row['Cross_Hemi']:>6} ({row['Cross_Hemi_Pct']:>5.1f}%) "
          f"{row['Both_Crossed']:>6} ({row['Both_Crossed_Pct']:>5.1f}%)")

# Calculate key statistics for Full model
full_rest = error_df[(error_df['Model'] == 'Full') & (error_df['Data'] == 'Rest')].iloc[0]
full_task = error_df[(error_df['Model'] == 'Full') & (error_df['Data'] == 'Task')].iloc[0]

cross_boundary_rest = full_rest['Cross_Network_Pct'] + full_rest['Both_Crossed_Pct']
cross_boundary_task = full_task['Cross_Network_Pct'] + full_task['Both_Crossed_Pct']

print("\n" + "="*110)
print("KEY FINDING: Error Distribution Challenges Network Homogeneity Assumptions")
print("="*110)
print(f"\n• Only {full_rest['Within_Network_Pct']:.1f}% (Rest) / {full_task['Within_Network_Pct']:.1f}% (Task) of errors occur within the same network")
print(f"• {cross_boundary_rest:.1f}% (Rest) / {cross_boundary_task:.1f}% (Task) of errors cross network boundaries")
print(f"• ~{full_rest['Both_Crossed_Pct']:.0f}% of errors cross BOTH hemisphere and network boundaries")
print(f"\n→ This suggests regions within the same network do NOT have more similar fingerprints")
print(f"→ The multinomial model distributes errors broadly rather than concentrating within networks")
print("="*110)

ERROR TAXONOMY: Where Do Classification Errors Occur?

Model    Data     Errors     Rate     Within-Net      Cross-Net     Cross-Hemi           Both
----------------------------------------------------------------------------------------------------
Full     Rest       3943    7.6%    592 ( 15.0%)   1319 ( 33.5%)    868 ( 22.0%)   1164 ( 29.5%)
Full     Task       4993   10.8%    849 ( 17.0%)   1687 ( 33.8%)   1035 ( 20.7%)   1422 ( 28.5%)
LH       Rest       2126    8.2%    699 ( 32.9%)   1427 ( 67.1%)      0 (  0.0%)      0 (  0.0%)
LH       Task       3032   13.1%    996 ( 32.8%)   2036 ( 67.2%)      0 (  0.0%)      0 (  0.0%)
RH       Rest       2246    8.6%    687 ( 30.6%)   1559 ( 69.4%)      0 (  0.0%)      0 (  0.0%)
RH       Task       3177   13.7%    949 ( 29.9%)   2228 ( 70.1%)      0 (  0.0%)      0 (  0.0%)

KEY FINDING: Error Distribution Challenges Network Homogeneity Assumptions

• Only 15.0% (Rest) / 17.0% (Task) of errors occur within the same network
• 63.0% (Rest) /

In [49]:
import plotly.graph_objects as go
import numpy as np

def plot_error_taxonomy_publication(error_df):
    # 1. Prepare Data
    x_labels = [f"<b>{row['Model']}</b><br>{row['Data']}" for _, row in error_df.iterrows()]
    
    # Categories ordered for logical stacking (bottom to top)
    categories = [
        ('Within_Network', 'Within Network (Same Hemi)', '#2b8cbe'),  # Dark Blue
        ('Cross_Network', 'Cross Network (Same Hemi)', '#a6bddb'),   # Light Blue
        ('Cross_Hemi', 'Cross Hemi (Same Net)', '#e6550d'),         # Dark Orange
        ('Both_Crossed', 'Both Crossed (Diff Hemi/Net)', '#fdae6b')  # Light Orange
    ]

    fig = go.Figure()

    # 2. Add Stacked Bar Traces
    for col, label, color in categories:
        pcts = (error_df[col] / error_df['Total_Errors'] * 100)
        
        fig.add_trace(go.Bar(
            name=label,
            x=x_labels,
            y=pcts,
            marker_color=color,
            marker_line=dict(color='rgba(0,0,0,0.5)', width=0.8),
            # Precise labeling: % and raw N
            text=[f"<b>{p:.1f}%</b><br>N={int(c)}" if p > 6 else "" 
                  for p, c in zip(pcts, error_df[col])],
            textposition='inside',
            insidetextanchor='middle',
            textfont=dict(size=11, color='white' if 'Within_Network' in col else 'black'),
            hovertemplate="<b>%{x}</b><br>%{y:.1f}% of total errors<extra></extra>"
        ))

    # 3. Add Model Statistics above bars
    for i, (_, row) in enumerate(error_df.iterrows()):
        fig.add_annotation(
            x=x_labels[i], y=101,
            text=f"Total Errors: {int(row['Total_Errors'])}<br><b>Rate: {row['Error_Rate']:.2%}</b>",
            showarrow=False, font=dict(size=10, color="#444"), align="center", yshift=15
        )

    # 4. NETWORK-LEVEL ANNOTATIONS (LEFT BRACKET)
    # Highlight the distinction between within and cross network
    fig.add_shape(type="line", x0=-0.6, y0=0.5, x1=-0.6, y1=15, line=dict(color="#2b8cbe", width=3))
    fig.add_annotation(x=-0.65, y=7.5, text="Within-Net", textangle=-90, showarrow=False, 
                       font=dict(color="#2b8cbe", size=11, weight="bold"), xanchor="right")
    
    fig.add_shape(type="line", x0=-0.6, y0=16, x1=-0.6, y1=65, line=dict(color="#a6bddb", width=3))
    fig.add_annotation(x=-0.65, y=40, text="Cross-Net", textangle=-90, showarrow=False, 
                       font=dict(color="#6b8ba4", size=11, weight="bold"), xanchor="right")

    # 5. HEMISPHERE ELIMINATION ANNOTATION (RIGHT BRACKET)
    fig.add_vline(x=1.5, line_dash="dash", line_color="rgba(0,0,0,0.2)", line_width=1)
    fig.add_vline(x=3.5, line_dash="dash", line_color="rgba(0,0,0,0.2)", line_width=1)

    fig.add_shape(
        type="line", x0=1.7, y0=122, x1=5.3, y1=122,
        line=dict(color="#e6550d", width=2)
    )
    fig.add_annotation(
        x=3.5, y=125,
        text="HEMISPHERE-RELATED ERRORS ELIMINATED",
        showarrow=False,
        font=dict(size=12, color="#e6550d", family="Arial Black"),
    )

    # 6. Global Layout
    fig.update_layout(
        title=dict(
            text="Classification Error Taxonomy: Constraints on Model Performance",
            font=dict(size=22, family="Arial Black"), x=0.5, y=0.95
        ),
        barmode='stack',
        yaxis=dict(
            title="Proportion of Errors (%)", 
            range=[-5, 140], 
            tickfont=dict(size=12),
            gridcolor='rgba(0,0,0,0.05)'
        ),
        xaxis=dict(title="Model Comparison (Full vs. Hemi-Specific)", tickfont=dict(size=11)),
        legend=dict(
            title="<b>Error Category</b>", 
            orientation="v", 
            yanchor="top", y=0.9, 
            xanchor="left", x=1.02,
            bordercolor="rgba(0,0,0,0.1)", borderwidth=1
        ),
        template="plotly_white",
        width=1300, height=800,
        margin=dict(t=120, b=100, l=120, r=250)
    )

    return fig

fig = plot_error_taxonomy_publication(error_df)
fig.show()

---
## 5. Network-Level Error Patterns

We now examine which functional networks confuse with which others. This analysis aggregates the 232×232 confusion matrix into an 8×8 network-level view.

**What to look for**:
- Are errors concentrated in specific network pairs (suggesting interpretable confusion patterns)?
- Does the task condition change which networks confuse with each other?
- Which networks show the largest error increases during task engagement?

In [51]:
def aggregate_to_network_sum(region_matrix, region_info):
    """Aggregate region-level confusion matrix to network-level absolute counts."""
    net_labels = region_info['major_network'].values
    df = pd.DataFrame(region_matrix, index=net_labels, columns=net_labels)
    
    row_grouped = df.groupby(level=0).sum()
    full_grouped = row_grouped.transpose().groupby(level=0).sum().transpose()
    
    net_matrix = full_grouped.reindex(index=NETWORKS, columns=NETWORKS).fillna(0)
    return net_matrix.values

# Process matrices for all models
full_rest_net = aggregate_to_network_sum(confusion_matrix_cv_full, region_info_full)
full_task_net = aggregate_to_network_sum(confusion_matrix_task_full, region_info_full)
lh_rest_net = aggregate_to_network_sum(confusion_matrix_cv_lh, region_info_lh)
lh_task_net = aggregate_to_network_sum(confusion_matrix_task_lh, region_info_lh)
rh_rest_net = aggregate_to_network_sum(confusion_matrix_cv_rh, region_info_rh)
rh_task_net = aggregate_to_network_sum(confusion_matrix_task_rh, region_info_rh)

# Organize for plotting: [Full, Left, Right] × [Rest, Task, Difference]
matrix_data = [
    [full_rest_net, full_task_net, (full_task_net - full_rest_net)],
    [lh_rest_net, lh_task_net, (lh_task_net - lh_rest_net)],
    [rh_rest_net, rh_task_net, (rh_task_net - rh_rest_net)]
]

# Define color schemes
colorscale_seq = [[0, '#f7fbff'], [1, '#2b8cbe']]  # Sequential blue
colorscale_div = [[0, '#2b8cbe'], [0.5, '#ffffff'], [1, '#e6550d']]  # Diverging blue-orange

# Create 3×3 heatmap grid
rows_labels = ['Full', 'Left', 'Right']
cols_labels = ['Rest (CV)', 'Task (Test)', 'Difference (T-R)']

fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=[f"<b>{r}: {c}</b>" for r in rows_labels for c in cols_labels],
    horizontal_spacing=0.12, 
    vertical_spacing=0.12
)

for i in range(3):
    for j in range(3):
        data = matrix_data[i][j].copy()
        mask = np.eye(len(NETWORKS), dtype=bool)
        data_masked = np.where(mask, None, data)  # Mask diagonal (correct classifications)
        
        if j < 2:  # Rest or Task
            colorscale = colorscale_seq
            valid_vals = data_masked[data_masked != None]
            zmin, zmax = 0, np.nanmax(valid_vals) if len(valid_vals) > 0 else 1
        else:  # Difference
            colorscale = colorscale_div
            valid_vals = data_masked[data_masked != None].astype(float)
            limit = np.nanmax(np.abs(valid_vals)) if len(valid_vals) > 0 else 1
            zmin, zmax = -limit, limit

        fig.add_trace(
            go.Heatmap(
                z=data_masked, x=NETWORKS, y=NETWORKS,
                colorscale=colorscale, zmin=zmin, zmax=zmax,
                text=data_masked, texttemplate="%{text:.0f}", textfont={"size": 8},
                showscale=False,
                hovertemplate="True: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>"
            ),
            row=i+1, col=j+1
        )

fig.update_layout(
    title_text="<b>Network-Level Confusion Matrices: Off-Diagonal Error Counts</b><br>" +
               "<sup>Diagonal (correct classifications) masked. Orange in difference = increased errors during task.</sup>",
    template="plotly_white",
    height=1200, width=1400,
    margin=dict(l=120, r=50, t=120, b=100)
)

fig.update_xaxes(tickangle=45, tickfont=dict(size=9))
fig.update_yaxes(autorange="reversed", tickfont=dict(size=9))

fig.show()

In [12]:
# Quantify network-level error contributions

def calculate_network_error_rates(cm_rest, cm_task, region_info, n_subjects):
    """Calculate per-network error rates for rest and task conditions."""
    net_counts = region_info.groupby('major_network').size()
    
    results = []
    for net in NETWORKS:
        n_regions = net_counts.get(net, 0)
        if n_regions == 0:
            continue
            
        # Get indices for this network
        net_indices = region_info[region_info['major_network'] == net]['region_idx'].values
        
        # Sum off-diagonal errors (misclassifications of this network's regions)
        rest_errors = sum(cm_rest[idx, :].sum() - cm_rest[idx, idx] for idx in net_indices)
        task_errors = sum(cm_task[idx, :].sum() - cm_task[idx, idx] for idx in net_indices)
        
        total_samples = n_regions * n_subjects
        
        results.append({
            'Network': net,
            'N_Regions': n_regions,
            'Rest_Errors': int(rest_errors),
            'Task_Errors': int(task_errors),
            'Rest_Rate': rest_errors / total_samples,
            'Task_Rate': task_errors / total_samples,
            'Delta': (task_errors - rest_errors) / total_samples,
            'Pct_Increase': ((task_errors - rest_errors) / rest_errors * 100) if rest_errors > 0 else 0
        })
    
    return pd.DataFrame(results)

# Calculate for Full model
network_errors = calculate_network_error_rates(
    confusion_matrix_cv_full, confusion_matrix_task_full, 
    region_info_full, N_SUBJECTS
)

# Sort by task error rate
network_errors = network_errors.sort_values('Task_Rate', ascending=False)

print("="*100)
print("NETWORK-LEVEL ERROR RATES (Full Model)")
print("="*100)
print(f"\n{'Network':<28} {'Regions':>8} {'Rest Rate':>12} {'Task Rate':>12} {'Δ Rate':>10} {'% Increase':>12}")
print("-"*85)

for _, row in network_errors.iterrows():
    print(f"{row['Network']:<28} {row['N_Regions']:>8} {row['Rest_Rate']:>11.1%} "
          f"{row['Task_Rate']:>11.1%} {row['Delta']:>+9.1%} {row['Pct_Increase']:>+11.1f}%")

# Identify key patterns
highest_task = network_errors.iloc[0]
highest_increase = network_errors.loc[network_errors['Pct_Increase'].idxmax()]

print("\n" + "="*100)
print("KEY PATTERNS:")
print("="*100)
print(f"• Highest task error rate: {highest_task['Network']} ({highest_task['Task_Rate']:.1%})")
print(f"• Largest relative increase: {highest_increase['Network']} (+{highest_increase['Pct_Increase']:.0f}%)")
print(f"• Subcortical structures show highest error rates, consistent with their role in task engagement")
print("="*100)

NETWORK-LEVEL ERROR RATES (Full Model)

Network                       Regions    Rest Rate    Task Rate     Δ Rate   % Increase
-------------------------------------------------------------------------------------
Subcortical                        32       16.9%       23.0%     +6.1%       +35.9%
Limbic                             14       14.1%       14.3%     +0.2%        +1.1%
Salience/Ventral Attention         26        6.8%        9.0%     +2.1%       +31.2%
Default                            43        6.5%        7.8%     +1.3%       +19.7%
Control                            37        5.7%        7.3%     +1.6%       +28.3%
Somatomotor                        34        4.2%        6.4%     +2.1%       +50.2%
Dorsal Attention                   22        6.1%        5.4%     -0.7%       -11.4%
Visual                             24        3.1%        4.9%     +1.8%       +59.6%

KEY PATTERNS:
• Highest task error rate: Subcortical (23.0%)
• Largest relative increase: Visual (+60%)
•

---
## 6. Hemispheric Flow Analysis

Sankey diagrams visualize the flow of classification errors from true labels (left) through hemispheric routing (middle) to predicted labels (right). This reveals:

1. Which networks contribute most errors
2. Whether errors preferentially stay within or cross hemispheres
3. Which network pairs show the strongest confusion patterns

In [13]:
def create_sankey_data(true_labels, pred_labels, region_info):
    """Prepare data for Sankey diagram showing error flows."""
    
    # Get only errors
    err_mask = true_labels != pred_labels
    true_err = true_labels[err_mask]
    pred_err = pred_labels[err_mask]
    
    lookup = region_info.set_index('region_idx')
    
    # Create error DataFrame
    df = pd.DataFrame({
        'true_net': [lookup.loc[i, 'major_network'] for i in true_err],
        'true_hemi': [lookup.loc[i, 'hemisphere'] for i in true_err],
        'pred_net': [lookup.loc[i, 'major_network'] for i in pred_err],
        'pred_hemi': [lookup.loc[i, 'hemisphere'] for i in pred_err]
    })
    
    # Network colors (consistent with standard neuroimaging palettes)
    net_colors = {
        'Control': 'rgb(230, 148, 34)', 'Default': 'rgb(205, 62, 78)',
        'Dorsal Attention': 'rgb(0, 118, 14)', 'Limbic': 'rgb(220, 248, 164)',
        'Salience/Ventral Attention': 'rgb(196, 58, 250)', 'Somatomotor': 'rgb(70, 130, 180)',
        'Subcortical': 'rgb(12, 123, 174)', 'Visual': 'rgb(120, 18, 134)'
    }
    
    # Build node lists
    source_nets = sorted(df['true_net'].unique())
    hemis = ['LEFT', 'RIGHT']
    target_nets = sorted(df['pred_net'].unique())
    
    # Node indices: [Source Networks (0-7)] + [Hemispheres (8-11)] + [Target Networks (12-19)]
    node_labels = [n.upper() for n in source_nets] + hemis + hemis + [n.upper() for n in target_nets]
    node_colors = ([net_colors.get(n, '#888888') for n in source_nets] + 
                   ['#505050', '#505050', '#505050', '#505050'] +
                   [net_colors.get(n, '#888888') for n in target_nets])
    
    # Build links
    sources, targets, values, link_colors = [], [], [], []
    
    n_source = len(source_nets)
    n_hemi = 4  # 2 source hemis + 2 target hemis
    
    # Stage 1: Source Network → Source Hemisphere
    for i, net in enumerate(source_nets):
        for j, hemi in enumerate(['left', 'right']):
            count = len(df[(df['true_net'] == net) & (df['true_hemi'] == hemi)])
            if count > 0:
                sources.append(i)
                targets.append(n_source + j)  # 8 or 9
                values.append(count)
                link_colors.append(net_colors.get(net, '#888888').replace('rgb', 'rgba').replace(')', ', 0.3)'))
    
    # Stage 2: Source Hemisphere → Target Hemisphere
    for j_src, hemi_src in enumerate(['left', 'right']):
        for j_tgt, hemi_tgt in enumerate(['left', 'right']):
            count = len(df[(df['true_hemi'] == hemi_src) & (df['pred_hemi'] == hemi_tgt)])
            if count > 0:
                sources.append(n_source + j_src)  # 8 or 9
                targets.append(n_source + 2 + j_tgt)  # 10 or 11
                values.append(count)
                link_colors.append('rgba(211, 211, 211, 0.4)')
    
    # Stage 3: Target Hemisphere → Target Network
    for j, hemi in enumerate(['left', 'right']):
        for k, net in enumerate(target_nets):
            count = len(df[(df['pred_hemi'] == hemi) & (df['pred_net'] == net)])
            if count > 0:
                sources.append(n_source + 2 + j)  # 10 or 11
                targets.append(n_source + n_hemi + k)  # 12+
                values.append(count)
                link_colors.append(net_colors.get(net, '#888888').replace('rgb', 'rgba').replace(')', ', 0.3)'))
    
    return node_labels, node_colors, sources, targets, values, link_colors

# Create Sankey for task condition (where errors matter most)
node_labels, node_colors, sources, targets, values, link_colors = create_sankey_data(
    task_true_labels_full, task_predictions_full, region_info_full
)

fig = go.Figure(data=[go.Sankey(
    arrangement='fixed',
    node=dict(
        pad=30, thickness=20,
        line=dict(color='white', width=1),
        label=node_labels,
        color=node_colors
    ),
    link=dict(
        source=sources, target=targets, value=values,
        color=link_colors,
        hovertemplate="<b>%{source.label} → %{target.label}</b><br>Count: %{value}<extra></extra>"
    )
)])

fig.update_layout(
    title=dict(
        text="<b>Error Flow Analysis: Task Condition (Full Model)</b><br>" +
             "<sup>True Network → Hemisphere → Predicted Network</sup>",
        x=0.5, font=dict(size=18)
    ),
    font=dict(size=11),
    height=700, width=1200,
    template='simple_white'
)

fig.show()

# Print summary statistics
err_mask = task_true_labels_full != task_predictions_full
total_errors = err_mask.sum()
lookup = region_info_full.set_index('region_idx')

df_errors = pd.DataFrame({
    'true_hemi': [lookup.loc[i, 'hemisphere'] for i in task_true_labels_full[err_mask]],
    'pred_hemi': [lookup.loc[i, 'hemisphere'] for i in task_predictions_full[err_mask]]
})

within_hemi = (df_errors['true_hemi'] == df_errors['pred_hemi']).sum()
cross_hemi = (df_errors['true_hemi'] != df_errors['pred_hemi']).sum()

print(f"\nHemispheric Error Distribution (Task):")
print(f"  Within-hemisphere errors: {within_hemi:,} ({within_hemi/total_errors:.1%})")
print(f"  Cross-hemisphere errors:  {cross_hemi:,} ({cross_hemi/total_errors:.1%})")


Hemispheric Error Distribution (Task):
  Within-hemisphere errors: 2,536 (50.8%)
  Cross-hemisphere errors:  2,457 (49.2%)


In [46]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# 1. Create a consistent Color Mapping
# Sorting once to assign colors based on Task_Rate magnitude
df_color_base = network_errors.sort_values('Task_Rate', ascending=False)
palette = ['#E63946', '#F4A261', '#E9C46A', '#2A9D8F', '#264653', '#A8DADC', '#457B9D', '#1D3557']
network_colors = {net: palette[i % len(palette)] for i, net in enumerate(df_color_base['Network'])}

# 2. Setup Subplots
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("Rest-to-Task Error Transition", "Task-Evoked Increase in Error (Δ)"),
    horizontal_spacing=0.25
)

# --- PLOT 1: SLOPE CHART (Left) ---
# Sorting by Task_Rate for the visual flow of lines
slope_df = network_errors.sort_values('Task_Rate', ascending=False)

for _, row in slope_df.iterrows():
    color = network_colors[row['Network']]
    
    fig.add_trace(
        go.Scatter(
            x=['Rest', 'Task'],
            y=[row['Rest_Rate'], row['Task_Rate']],
            mode='lines+markers+text',
            name=row['Network'],
            line=dict(color=color, width=3),
            marker=dict(size=12, color=color),
            # Precise labeling to prevent overlap
            text=[f"{row['Network']}  {row['Rest_Rate']:.1%}", f"{row['Task_Rate']:.1%}"],
            textposition=["middle left", "middle right"],
            textfont=dict(size=11, color=color),
            hovertemplate=f"<b>{row['Network']}</b><br>Rest: {row['Rest_Rate']:.1%}<br>Task: {row['Task_Rate']:.1%}<extra></extra>"
        ),
        row=1, col=1
    )

# --- PLOT 2: DELTA BAR CHART (Right) ---
# Sorting by Delta (magnitude of change)
delta_df = network_errors.sort_values('Delta', ascending=True)
# Retrieve colors from our map to match Plot 1
bar_colors = [network_colors[net] for net in delta_df['Network']]

fig.add_trace(
    go.Bar(
        x=delta_df['Delta'],
        y=delta_df['Network'],
        orientation='h',
        marker_color=bar_colors,
        text=[f"{val:+.1%}" for val in delta_df['Delta']],
        textposition='outside',
        textfont=dict(size=11, weight='bold'),
        hovertemplate="<b>%{y}</b><br>Δ Change: %{x:+.1%}<extra></extra>"
    ),
    row=1, col=2
)

# --- GLOBAL LAYOUT & STYLING ---
fig.update_layout(
    height=700, width=1400, 
    title_text="Network-Level Functional Stability Analysis",
    title_font_size=24,
    template="plotly_white",
    showlegend=False,
    margin=dict(l=200, r=80, t=100, b=50) # Increased left margin for long network names
)

# Formatting Slope Chart Axis
fig.update_xaxes(range=[-0.6, 1.6], showgrid=False, row=1, col=1)
fig.update_yaxes(title_text="Classification Error Rate", tickformat=".0%", row=1, col=1)

# Formatting Bar Chart Axis (Start from -2% as requested)
fig.update_xaxes(
    title_text="Absolute Change in Error (%)", 
    tickformat=".0%", 
    range=[-0.02, 0.08], 
    row=1, col=2
)
fig.update_yaxes(showticklabels=True, row=1, col=2)

fig.show()

---
## 7. The Case Against Multinomial Classification

This section synthesizes the evidence from our error analysis to motivate the use of One-vs-Rest (OvR) and One-vs-One (OvO) classification strategies.

### Theoretical Limitations of Multinomial for 232-Class Problems

Multinomial logistic regression computes a single softmax over all 232 classes simultaneously:

$$P(y=k|\mathbf{x}) = \frac{\exp(\mathbf{w}_k^T \mathbf{x})}{\sum_{j=1}^{232} \exp(\mathbf{w}_j^T \mathbf{x})}$$

This creates several challenges:

1. **Probability dilution**: With 232 competing classes, even confident predictions have low absolute probabilities
2. **Global competition**: Each class must be distinguishable from ALL other classes simultaneously
3. **Single decision boundary**: No flexibility to use different features for different comparisons
4. **Asymmetric class relationships**: Some region pairs are inherently more similar than others

In [15]:
# Analyze prediction confidence distribution

def analyze_confidence(probabilities, true_labels, pred_labels, condition_name):
    """Analyze the confidence distribution of predictions."""
    
    # Get max probability for each prediction
    max_probs = probabilities.max(axis=1)
    
    # Separate correct and incorrect predictions
    correct_mask = true_labels == pred_labels
    
    correct_conf = max_probs[correct_mask]
    incorrect_conf = max_probs[~correct_mask]
    
    # Get second-highest probability for errors (confusion strength)
    sorted_probs = np.sort(probabilities, axis=1)[:, ::-1]
    top2_diff = sorted_probs[:, 0] - sorted_probs[:, 1]
    
    return {
        'Condition': condition_name,
        'Mean_Conf_Correct': correct_conf.mean(),
        'Mean_Conf_Incorrect': incorrect_conf.mean(),
        'Conf_Gap': correct_conf.mean() - incorrect_conf.mean(),
        'Low_Conf_Correct': (correct_conf < 0.5).mean(),  # % of correct with <50% confidence
        'High_Conf_Incorrect': (incorrect_conf > 0.5).mean(),  # % of incorrect with >50% confidence
        'Mean_Top2_Gap': top2_diff.mean(),
        'Median_Max_Prob': np.median(max_probs)
    }

# Analyze both conditions
rest_conf = analyze_confidence(cv_probabilities_full, cv_true_labels_full, cv_predictions_full, 'Rest')
task_conf = analyze_confidence(task_probabilities_full, task_true_labels_full, task_predictions_full, 'Task')

conf_df = pd.DataFrame([rest_conf, task_conf])

print("="*100)
print("PREDICTION CONFIDENCE ANALYSIS: Evidence of Probability Dilution")
print("="*100)
print(f"\n{'Condition':<10} {'Correct Conf':>14} {'Incorrect Conf':>16} {'Gap':>10} {'Low-Conf Correct':>18} {'High-Conf Errors':>18}")
print("-"*90)

for _, row in conf_df.iterrows():
    print(f"{row['Condition']:<10} {row['Mean_Conf_Correct']:>13.1%} {row['Mean_Conf_Incorrect']:>15.1%} "
          f"{row['Conf_Gap']:>9.1%} {row['Low_Conf_Correct']:>17.1%} {row['High_Conf_Incorrect']:>17.1%}")

print("\n" + "="*100)
print("INTERPRETATION:")
print("="*100)
print(f"• Mean confidence for correct predictions: {rest_conf['Mean_Conf_Correct']:.1%} (Rest) / {task_conf['Mean_Conf_Correct']:.1%} (Task)")
print(f"• Even CORRECT predictions often have relatively low confidence due to 232-class competition")
print(f"• {rest_conf['Low_Conf_Correct']:.1%} of correct rest predictions have <50% confidence")
print(f"• This probability dilution is a fundamental limitation of multinomial softmax")
print("="*100)

PREDICTION CONFIDENCE ANALYSIS: Evidence of Probability Dilution

Condition    Correct Conf   Incorrect Conf        Gap   Low-Conf Correct   High-Conf Errors
------------------------------------------------------------------------------------------
Rest               87.1%           36.9%     50.2%              7.3%             23.2%
Task               84.7%           38.0%     46.7%              9.3%             25.2%

INTERPRETATION:
• Mean confidence for correct predictions: 87.1% (Rest) / 84.7% (Task)
• Even CORRECT predictions often have relatively low confidence due to 232-class competition
• 7.3% of correct rest predictions have <50% confidence
• This probability dilution is a fundamental limitation of multinomial softmax


In [42]:
# Visualize confidence distributions

fig = make_subplots(rows=1, cols=2, subplot_titles=['<b>Rest (CV)</b>', '<b>Task (Test)</b>'])

for i, (probs, true_labs, pred_labs, name) in enumerate([
    (cv_probabilities_full, cv_true_labels_full, cv_predictions_full, 'Rest'),
    (task_probabilities_full, task_true_labels_full, task_predictions_full, 'Task')
]):
    max_probs = probs.max(axis=1)
    correct_mask = true_labs == pred_labs
    
    fig.add_trace(
        go.Histogram(x=max_probs[correct_mask], name='Correct', opacity=0.7,
                     marker_color='#2b8cbe', nbinsx=50, histnorm='probability'),
        row=1, col=i+1
    )
    fig.add_trace(
        go.Histogram(x=max_probs[~correct_mask], name='Incorrect', opacity=0.7,
                     marker_color='#e6550d', nbinsx=50, histnorm='probability'),
        row=1, col=i+1
    )

fig.update_layout(
    title=dict(
        text="<b>Prediction Confidence Distributions</b><br>" +
             "<sup>Maximum predicted probability for correct vs. incorrect classifications</sup>",
        x=0.5, font=dict(size=16)
    ),
    template='simple_white',
    height=400, width=1000,
    barmode='overlay',
    showlegend=True,
    legend=dict(x=0.85, y=0.95)
)

fig.update_xaxes(title_text='Maximum Predicted Probability', range=[0, 1])
fig.update_yaxes(title_text='Proportion')

fig.show()

In [17]:
# Summary statistics table

print("\n" + "="*110)
print("SUMMARY: EVIDENCE FOR OvR/OvO APPROACHES")
print("="*110)

# Compile key findings
findings = [
    ("1. Cross-Boundary Errors", 
     f"{cross_boundary_task:.1f}% of task errors cross network boundaries",
     "OvR: Each region trained against ALL others, regardless of network"),
    
    ("2. Probability Dilution",
     f"Median max probability: {np.median(task_probabilities_full.max(axis=1)):.1%} across 232 classes",
     "OvR: Binary decisions yield calibrated probabilities"),
    
    ("3. Non-Uniform Error Distribution",
     f"Subcortical: {highest_task['Task_Rate']:.1%} error rate vs. Visual: {network_errors[network_errors['Network']=='Visual']['Task_Rate'].values[0]:.1%}",
     "OvO: Can learn region-specific decision boundaries"),
    
    ("4. Task-Induced Reorganization",
     f"All networks show {df_combined['Pct_Change'].min():.0f}%-{df_combined['Pct_Change'].max():.0f}% error increases",
     "OvR/OvO: Can capture differential sensitivity to task demands")
]

print(f"\n{'Finding':<30} {'Evidence':<55} {'Alternative Approach Advantage'}")
print("-"*130)

for finding, evidence, advantage in findings:
    print(f"{finding:<30} {evidence:<55} {advantage}")

print("\n" + "="*110)
print("CONCLUSION:")
print("="*110)
print("""
The multinomial logistic regression analysis reveals fundamental limitations when applied to 
232-class brain region classification:

• Errors are NOT concentrated within functional networks (~88% cross network boundaries)
• Prediction confidence is diluted across many competing classes
• Error patterns suggest the model lacks region-specific discriminative power

These findings motivate the use of decomposed classification strategies:

→ ONE-VS-REST (OvR): Train 232 binary classifiers, each distinguishing one region from all others.
  Advantage: Explicit positive/negative training for each region; interpretable binary probabilities.

→ ONE-VS-ONE (OvO): Train classifiers for each region pair, aggregate via voting.
  Advantage: Can learn optimal boundaries for similar region pairs; scales quadratically but 
  allows fine-grained distinctions.

The subsequent chapters will implement and compare these strategies using the same error-as-signal
framework established here.
""")
print("="*110)


SUMMARY: EVIDENCE FOR OvR/OvO APPROACHES

Finding                        Evidence                                                Alternative Approach Advantage
----------------------------------------------------------------------------------------------------------------------------------
1. Cross-Boundary Errors       62.3% of task errors cross network boundaries           OvR: Each region trained against ALL others, regardless of network
2. Probability Dilution        Median max probability: 92.2% across 232 classes        OvR: Binary decisions yield calibrated probabilities
3. Non-Uniform Error Distribution Subcortical: 23.0% error rate vs. Visual: 4.9%          OvO: Can learn region-specific decision boundaries
4. Task-Induced Reorganization All networks show 2%-62% error increases                OvR/OvO: Can capture differential sensitivity to task demands

CONCLUSION:

The multinomial logistic regression analysis reveals fundamental limitations when applied to 
232-class brain 

---
## 8. Conclusions

### Key Findings

1. **Multinomial Performance**: The Full (232) model achieves ~92% CV accuracy but drops to ~89% on task transfer, indicating sensitivity to state-dependent connectivity changes.

2. **Error Distribution**: Classification errors do NOT preferentially occur within functional networks. Only ~4-5% of errors stay within the same network and hemisphere, challenging assumptions about network-level fingerprint homogeneity.

3. **Network-Specific Patterns**: Subcortical structures show the highest error rates (~30%+ during task), consistent with their central role in task engagement and their dense connectivity patterns.

4. **Probability Dilution**: The 232-class softmax competition results in relatively low confidence even for correct predictions, limiting the interpretability of prediction confidence.

### Implications for Thesis

This analysis establishes the baseline against which OvR and OvO strategies will be compared. The error patterns identified here—particularly the cross-boundary distribution and network-specific rates—provide testable hypotheses for whether decomposed strategies can achieve:

- Higher overall accuracy through specialized binary decisions
- More interpretable error patterns aligned with functional organization
- Better calibrated confidence estimates for the "error-as-signal" framework

### Next Steps

1. Implement One-vs-Rest classification with the same data splits
2. Implement One-vs-One classification (computational considerations for N*(N-1)/2 classifiers)
3. Compare error patterns across all three strategies
4. Interpret task-induced errors through the lens of functional reorganization